# Fairness Audit: project lab notebook
**HON-2501-3 · Responsible AI** · Project, Part 1

<a href="https://colab.research.google.com/github/acintron-cc/hon-2501-03/blob/main/notebooks/fairness_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook you collect the evidence for your evaluation report (Part 2 of the project). You don't write any code: you type values into form fields and press ▶.

**How to use this notebook**
1. **File → Save a copy in Drive**, so your work is kept.
2. Run the cells one at a time with ▶, from the top.
3. When a cell has a form field (a dropdown, a slider or a text box), change the value there and press ▶ on that cell.
4. Answer each **For your report** question in the empty text cell under it (double-click the cell to type).

> **Gemini rule:** Ask Gemini to explain a cell or a number. Never ask it to rewrite or fix a cell.

| Part | You will |
|---|---|
| 1 · Scenario and dataset | choose your dataset and read the scenario |
| 2 · Base rates | write each group's P(label = 1 \| group) as a fraction and a decimal |
| 3 · Baseline model | read the model's rates and gaps |
| 4 · Reweighing | compute four weights by hand and retrain the model |
| 5 · Thresholds | choose which rate to equalize and set one cutoff per group |
| 6 · Comparison | download the table and chart of the three models |
| 7 · Who bears the errors | count the false positives and false negatives in each group |

## Setup
This cell installs Fairlearn, the fairness library we use, and loads the other tools. It prints Ready. when it is done (1 to 2 minutes the first time). If Colab warns "This notebook was not authored by Google", choose Run anyway.

In [ ]:
# @title Install and load the tools
!pip install -q fairlearn==0.13.0

import ast, math, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_positive_rate, count
from fairlearn.postprocessing import ThresholdOptimizer
warnings.filterwarnings("ignore")

DATA_URL = "https://raw.githubusercontent.com/acintron-cc/hon-2501-03/main/datasets/"
TEAL, ORANGE = "#0B6E84", "#A8401A"
RATES = ["selection rate", "TPR", "FPR", "accuracy"]
METRICS = {
    "n": count,
    "selection rate": selection_rate,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
    "accuracy": accuracy_score,
}
GAP_ROW = "gap (larger − smaller)"
GAPS = {"selection rate": "selection-rate gap", "TPR": "TPR gap", "FPR": "FPR gap"}

def rnd(x, d=3):
    """Round half up, the way Excel does."""
    return math.floor(x * 10**d + 0.5 + 1e-9) / 10**d

def fmt(x, d=3):
    return f"{rnd(x, d):.{d}f}"

def group_line():
    print(f"Group A = {GROUPS[0]}, Group B = {GROUPS[1]}")

def two_way(rows):
    """Group x label counts with totals."""
    t = pd.crosstab(rows[GROUP], rows["label"]).reindex(index=GROUPS, columns=[1, 0], fill_value=0)
    t.columns = ["label = 1", "label = 0"]
    t["Total"] = t.sum(axis=1)
    t.loc["Total"] = t.sum()
    t.index.name = GROUP
    return t

def by_group(y_true, y_pred, sensitive):
    """Fairlearn's MetricFrame: each metric computed separately for each group."""
    mf = MetricFrame(metrics=METRICS, y_true=y_true, y_pred=y_pred, sensitive_features=sensitive)
    return mf.by_group.reindex(GROUPS)

def gap(col):
    """Gap = larger group value - smaller group value, naming the larger group."""
    a, b = col[GROUPS[0]], col[GROUPS[1]]
    g = rnd(abs(a - b))
    if g == 0:
        return "0.000 (equal)"
    return f"{g:.3f} ({GROUPS[0] if a > b else GROUPS[1]} higher)"

def gap_size(bg, c):
    return rnd(abs(bg.loc[GROUPS[0], c] - bg.loc[GROUPS[1], c]))

def rate_table(bg, cols=RATES):
    out = bg[cols].map(fmt)
    out.loc[GAP_ROW] = [gap(bg[c]) for c in cols]
    return out

def counts(rows, pred):
    y, p = rows["label"].to_numpy(), np.asarray(pred)
    return dict(TP=int(((y == 1) & (p == 1)).sum()), FN=int(((y == 1) & (p == 0)).sum()),
                FP=int(((y == 0) & (p == 1)).sum()), TN=int(((y == 0) & (p == 0)).sum()))

CALC_OPS = {ast.Add: lambda a, b: a + b, ast.Sub: lambda a, b: a - b,
            ast.Mult: lambda a, b: a * b, ast.Div: lambda a, b: a / b,
            ast.UAdd: lambda a: a, ast.USub: lambda a: -a}

def safe_calc(text):
    """Evaluate numbers with + - * / and parentheses only (no eval). Returns a number or a message."""
    friendly = ("The calculator only understands numbers, + - * / and parentheses, "
                "for example (343/1400) / (55/466). Please check what you typed and press ▶ again.")
    if not re.fullmatch(r"[0-9\s.+\-*/()]+", text or ""):
        return friendly
    def ev(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in CALC_OPS:
            return CALC_OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in CALC_OPS:
            return CALC_OPS[type(node.op)](ev(node.operand))
        raise ValueError
    try:
        return ev(ast.parse(text, mode="eval").body)
    except ZeroDivisionError:
        return "That divides by zero. Please check your fractions and press ▶ again."
    except (SyntaxError, ValueError):
        return friendly

def exact_weights():
    """Reference weights, P(label) / P(label | group), computed from the train-row fractions."""
    p1 = train["label"].mean()
    w = {}
    for g in GROUPS:
        pg = train.loc[train[GROUP] == g, "label"].mean()
        w[(g, 1)], w[(g, 0)] = p1 / pg, (1 - p1) / (1 - pg)
    return w

def your_weights():
    """The class notebook's four weight fields (Section 4.2)."""
    A, B = GROUPS
    v = globals()
    return {(A, 1): v.get("w_label1_groupA", 0.0), (A, 0): v.get("w_label0_groupA", 0.0),
            (B, 1): v.get("w_label1_groupB", 0.0), (B, 0): v.get("w_label0_groupB", 0.0)}

def check_weights(w):
    """For each group: (group, arithmetic, weighted rate of label = 1, passes?)."""
    p1 = train["label"].mean()
    out = []
    for g in GROUPS:
        t = train[train[GROUP] == g]
        n1 = int((t["label"] == 1).sum())
        n0 = len(t) - n1
        top = n1 * w[(g, 1)]
        bottom = top + n0 * w[(g, 0)]
        rate = top / bottom
        text = (f"({n1} × {w[(g, 1)]:.4f}) ÷ ({n1} × {w[(g, 1)]:.4f} + {n0} × {w[(g, 0)]:.4f})"
                f" = {fmt(top, 2)} ÷ {fmt(bottom, 2)} = {fmt(rate, 4)}")
        out.append((g, text, rate, abs(rate - p1) <= 0.005))
    return out

def print_check(w):
    n1, N = int((train["label"] == 1).sum()), len(train)
    print(f"P(label = 1) = {n1}/{N} = {fmt(n1 / N, 4)}\n")
    for g, text, rate, ok in check_weights(w):
        print(f"Weighted rate of label = 1, {g}:")
        print(f"  {text}")
        print(f"  Same as P(label = 1)? {'yes' if ok else 'recheck'}\n")

def fit_reweighed(w):
    """Retrain the original model type with each train row counting by its weight; predict the test rows."""
    rw = np.array([w[(g, y)] for g, y in zip(train[GROUP], train["label"])])
    m = new_model()
    m.fit(X_train, y_train, logisticregression__sample_weight=rw)
    return m.predict(X_test)

def fit_threshold(constraint):
    """Fairlearn's ThresholdOptimizer on the train rows; predictions for the train and test rows."""
    to = ThresholdOptimizer(estimator=new_model(), constraints=constraint, objective="accuracy_score",
                            prefit=False, predict_method="predict_proba")
    to.fit(X_train, y_train, sensitive_features=train[GROUP])
    return (np.asarray(to.predict(X_train, sensitive_features=train[GROUP], random_state=42)),
            np.asarray(to.predict(X_test, sensitive_features=test[GROUP], random_state=42)))

def before_after(before, after):
    """Rates and gaps before and after, side by side."""
    cols = {}
    for g in GROUPS:
        cols[f"{g}, before"] = before.loc[g, RATES].map(fmt)
        cols[f"{g}, after"] = after.loc[g, RATES].map(fmt)
    cols["gap, before"] = [gap(before[c]) for c in RATES]
    cols["gap, after"] = [gap(after[c]) for c in RATES]
    return pd.DataFrame(cols, index=RATES)

def threshold_game(cuts):
    """One cutoff per group on the model's score (test rows): predictions, rates and the results table."""
    pred = (test["score"] >= test[GROUP].map(cuts)).astype(int)
    bg = by_group(y_test, pred, test[GROUP])
    out = rate_table(bg)
    out.insert(0, "cutoff", [f"{cuts[g]:.2f}" for g in GROUPS] + [""])
    return pred, bg, out

def compare(models):
    """Rates and gaps of several models (test rows); gap cells colored against the first model."""
    first = list(models)[0]
    table, gaps = {}, {}
    for name, p in models.items():
        bg_m = by_group(y_test, p, test[GROUP])
        row = {f"{c}, {g}": fmt(bg_m.loc[g, c]) for c in GAPS for g in GROUPS}
        gaps[name] = {c: gap_size(bg_m, c) for c in GAPS}
        for c, label in GAPS.items():
            row[label] = gap(bg_m[c])
            if name != first:
                d = gaps[name][c] - gaps[first][c]
                row[label] += " ↓" if d < 0 else " ↑" if d > 0 else ""
        row["accuracy, all"] = fmt(accuracy_score(y_test, p))
        table[name] = row
    table = pd.DataFrame(table).T

    def color_gaps(t):
        styles = pd.DataFrame("", index=t.index, columns=t.columns)
        for name in list(models)[1:]:
            for c, label in GAPS.items():
                d = gaps[name][c] - gaps[first][c]
                if d != 0:
                    styles.loc[name, label] = f"background-color: {TEAL if d < 0 else ORANGE}; color: white; font-weight: bold"
        return styles

    return table, gaps, table.style.apply(color_gaps, axis=None)

def plot_gaps(gaps):
    """One panel per gap; the first model is the outlined reference bar."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)
    names = list(gaps)
    top = max(max(gaps[m].values()) for m in names)
    for ax, (c, label) in zip(axes, GAPS.items()):
        for i, m in enumerate(names):
            v, d = gaps[m][c], gaps[m][c] - gaps[names[0]][c]
            if i == 0:
                ax.bar(i, v, 0.6, color="white", edgecolor=TEAL, linewidth=2)
                note = names[0]
            else:
                ax.bar(i, v, 0.6, color=ORANGE if d > 0 else TEAL)
                note = "larger" if d > 0 else "smaller" if d < 0 else "same"
            ax.text(i, v + top * 0.03, f"{v:.3f}\n{note}", ha="center", va="bottom", fontsize=9)
        ax.set_xticks(range(len(names)), [m.split(" (")[0] for m in names])
        ax.set_title(label)
        ax.set_xlabel("model")
        ax.grid(axis="y", alpha=0.3)
    axes[0].set_ylabel("gap (larger − smaller)")
    axes[0].set_ylim(0, top * 1.35 + 0.01)
    fig.suptitle(f"{DATASET}: gaps between {GROUPS[0]} and {GROUPS[1]}, test rows (smaller = more equal)")
    fig.tight_layout()
    return fig

GAP_LEGEND = "Teal with ↓: the gap is smaller than the original model's.  Orange with ↑: the gap is larger.  No color: the same gap."

import textwrap

SCENARIOS = {
    "german_credit": dict(
        label1="good credit risk, so the loan is approved (favorable)",
        text=["A regional bank wants to use the model to approve small personal loans automatically.",
              "Loan officers see the model's decision, and each applicant receives an approval or a denial.",
              "The people affected are the loan applicants, and the bank, which loses money on loans that are not repaid.",
              "A false positive approves an applicant who is a bad credit risk, who may then struggle to repay the loan.",
              "A false negative denies an applicant who is a good credit risk, who loses credit they would have repaid."],
        harm="FN", who={"female": "female applicants", "male": "male applicants"},
        harm_phrase="who are good credit risks were denied a loan"),
    "bank": dict(
        label1="subscribed, so the client is called (favorable)",
        text=["A bank's marketing team wants to use the model to decide whom to call with a term-deposit offer.",
              "The marketing team sees the model's list of clients to call.",
              "The people affected are the bank's clients, who get a call or don't.",
              "A false positive calls a client who won't subscribe, which wastes the client's time and the bank's.",
              "A false negative skips a client who would have subscribed, who never hears about an offer they wanted."],
        harm="FN", who={"under 25 or 60+": "clients under 25 or 60+", "25-59": "clients aged 25-59"},
        harm_phrase="who would have subscribed were not called"),
    "adult": dict(
        label1="income over $50K, so the person is pre-approved (favorable)",
        text=["A credit-card company wants to use the model to pre-approve people for a premium card, based on predicted income above $50K.",
              "The company's card team sees the result, and each person sees only whether they got an offer.",
              "The people affected are the potential customers, who get the offer and its benefits or don't.",
              "A false positive pre-approves someone who earns $50K or less, who may take on a card they can't afford.",
              "A false negative passes over someone who earns more than $50K, who misses the offer."],
        harm="FN", who={"Female": "women", "Male": "men"},
        harm_phrase="who earn more than $50K were not pre-approved"),
    "compas": dict(
        label1="flagged high risk of rearrest (harmful)",
        text=["A county court wants to use the model to recommend whether to hold defendants in jail before trial.",
              "Judges see the model's recommendation before they decide.",
              "The people affected are the defendants, and the public.",
              "A false positive flags a defendant who would not have been rearrested, who may be held in jail without need.",
              "A false negative clears a defendant who was later rearrested, which may put the public at risk."],
        harm="FP", who={"African-American": "African-American defendants", "Caucasian": "Caucasian defendants"},
        harm_phrase="who were not rearrested were flagged"),
}

def number_check(value, truth, tol):
    """yes / recheck for a number typed in a form field. Never shows the correct value."""
    if value == 0:
        return "recheck (empty)"
    return "yes" if abs(value - truth) <= tol + 1e-12 else "recheck"

def fraction_check(text, truth):
    """yes / recheck for a fraction typed as text, evaluated with the safe calculator."""
    if not str(text).strip():
        return "recheck (empty)"
    value = safe_calc(str(text))
    if isinstance(value, str):
        return "recheck (not a fraction)"
    return "yes" if abs(value - truth) < 1e-9 else "recheck"

def your_audit_weights():
    """Part 4's four weight fields."""
    A, B = GROUPS
    v = globals()
    return {(A, 1): v.get("A1_weight", 0.0), (A, 0): v.get("A0_weight", 0.0),
            (B, 1): v.get("B1_weight", 0.0), (B, 0): v.get("B0_weight", 0.0)}

def your_models():
    """The models built so far: the original, your reweighed model (Part 4), your threshold model (Part 5)."""
    v = globals()
    models, missing = {"original": test["predicted"].to_numpy()}, []
    if v.get("pred_rw_yours") is None:
        missing.append("Part 4 (retrain with your weights)")
    else:
        models["reweighed (yours)"] = v["pred_rw_yours"]
    if v.get("pred_game") is None:
        missing.append("Part 5 (your cutoffs)")
    else:
        models["threshold (yours)"] = np.asarray(v["pred_game"])
    return models, missing

print("Ready.")

## Part 1 · Scenario and dataset
**Choose a dataset you did not use in class** (in class you used adult or compas). Pick it from the dropdown and press ▶ to read your scenario.

In [ ]:
# @title Choose your dataset and read the scenario
DATASET = "german_credit"  # @param ["adult", "compas", "german_credit", "bank"]

CONFIG = {
    "adult": dict(target="class", group="sex", groups=["Female", "Male"],
                  drop=["fnlwgt", "education", "sex", "race"],
                  meaning="income over $50K", favorable=True),
    "compas": dict(target="two_year_recid", group="race_group", groups=["African-American", "Caucasian"],
                   drop=["race", "sex", "c_charge_desc"],
                   meaning="rearrested within 2 years", favorable=False),
    "german_credit": dict(target="credit-risk", group="sex", groups=["female", "male"],
                          drop=["sex", "marital_status", "age"],
                          meaning="good credit risk", favorable=True),
    "bank": dict(target="deposit", group="age_group", groups=["under 25 or 60+", "25-59"],
                 drop=["age"], meaning="subscribed to a deposit", favorable=True),
}
cfg = CONFIG[DATASET]
GROUP, GROUPS = cfg["group"], cfg["groups"]

local = os.environ.get("FAIRNESS_LOCAL_DATA")   # instructor's offline tests only
df = pd.read_csv(os.path.join(local, f"{DATASET}_classroom.csv") if local else DATA_URL + f"{DATASET}_classroom.csv")
df = df[df[GROUP].isin(GROUPS)].reset_index(drop=True)
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

# The model: a logistic regression that never sees the group column.
never = ["label", "split", "score", "predicted", "predicted_reweighed", "predicted_threshold",
         cfg["target"], GROUP] + cfg["drop"]
FEATURES = [c for c in df.columns if c not in never]
CATEGORICAL = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]

def new_model():
    return make_pipeline(
        make_column_transformer((OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
                                (StandardScaler(), NUMERIC)),
        LogisticRegression(max_iter=1000))

X_train, y_train = train[FEATURES], train["label"]
X_test, y_test = test[FEATURES], test["label"]
pred_ref_rw = pred_ref_to = pred_ref_to_train = None   # reference models, built in Sections 4 and 5

print(f"Dataset: {DATASET}")
print(f"label = 1 means: {cfg['meaning']}. This is a {'favorable' if cfg['favorable'] else 'HARMFUL'} outcome for the person.")
print(f"Group column: {GROUP}")
group_line()
if DATASET == "compas":
    print("Rows with race_group = Other are left out.")
print(f"Rows: {len(train):,} train, {len(test):,} test")

pred_rw_yours = pred_game = None   # your models, built in Parts 4 and 5

sc = SCENARIOS[DATASET]
print(f"\nlabel = 1 in this scenario: {sc['label1']}")
print("\nScenario")
print(textwrap.fill(" ".join(sc["text"]), 100))

**For your report:** In one sentence, what decision will the model make, and for whom?

## Part 2 · Base rates (all rows)
The base rate is P(label = 1 | group): how often each group has label = 1, before any model is involved. Use the table below to work out each group's base rate.

In [ ]:
# @title Group × label counts, all rows
group_line()
display(two_way(df))

Type each group's base rate as a fraction (for example 55/466) and as a decimal with 4 decimals (for example 0.1180). Then press ▶.

In [ ]:
# @title Your base rates
P_label1_given_groupA_fraction = ""  # @param {type:"string"}
P_label1_given_groupA_decimal = 0.0  # @param {type:"number"}
P_label1_given_groupB_fraction = ""  # @param {type:"string"}
P_label1_given_groupB_decimal = 0.0  # @param {type:"number"}

group_line()

In [ ]:
# @title Check your base rates
group_line()
fields = {"A": (P_label1_given_groupA_fraction, P_label1_given_groupA_decimal),
          "B": (P_label1_given_groupB_fraction, P_label1_given_groupB_decimal)}
rows = []
for g, letter in zip(GROUPS, "AB"):
    truth = df.loc[df[GROUP] == g, "label"].mean()
    frac, dec = fields[letter]
    rows.append({"field": f"P_label1_given_group{letter}_fraction", "stands for": f"P(label = 1 | {g})",
                 "check": fraction_check(frac, truth)})
    rows.append({"field": f"P_label1_given_group{letter}_decimal", "stands for": f"P(label = 1 | {g})",
                 "check": number_check(dec, truth, 0.0001)})
display(pd.DataFrame(rows).set_index("field"))

**For your report:** Which group has the higher base rate, and by how much? What might explain the difference? Name one possible cause in the world or in how the data was collected.

## Part 3 · Baseline model (test rows)
A confusion matrix counts the test rows by what really happened (actual) and what the model said (predicted). TP = true positive, FN = false negative, FP = false positive, TN = true negative.

In [ ]:
# @title Confusion matrix per group, test rows
for g in GROUPS:
    t = test[test[GROUP] == g]
    c = counts(t, t["predicted"])
    cm = pd.DataFrame([[f"TP = {c['TP']}", f"FN = {c['FN']}"], [f"FP = {c['FP']}", f"TN = {c['TN']}"]],
                      index=["actual = 1", "actual = 0"], columns=["predicted = 1", "predicted = 0"])
    print(f"\n{g} ({len(t)} test rows)")
    display(cm)

Each rate is a fraction built from these counts. The gap is the larger group's value minus the smaller group's value.

In [ ]:
# @title Rates and gaps, test rows
bg = by_group(y_test, test["predicted"], test[GROUP])
rates = bg[RATES].map(fmt)
rates.insert(0, "n", bg["n"].astype(int))
print("Rates by group (MetricFrame):")
display(rates)

fractions, selected = {}, {}
for g in GROUPS:
    t = test[test[GROUP] == g]
    c, n = counts(t, t["predicted"]), len(t)
    parts = {"selection rate": (c["TP"] + c["FP"], n), "TPR": (c["TP"], c["TP"] + c["FN"]),
             "FPR": (c["FP"], c["FP"] + c["TN"]), "accuracy": (c["TP"] + c["TN"], n)}
    fractions[g] = {k: f"{k} = {a}/{b} = {fmt(a / b)}" for k, (a, b) in parts.items()}
    selected[g] = parts["selection rate"]
print("\nThe same rates as fractions of the counts above:")
display(pd.DataFrame(fractions).T)

print("\nGaps:")
display(pd.DataFrame([[gap(bg[c]) for c in RATES]], index=[GAP_ROW], columns=RATES))

small, large = sorted((a / b, f"{a}/{b}") for a, b in selected.values())
ratio = small[0] / large[0]
print(f"\n80% rule: smaller selection rate ÷ larger = ({small[1]}) ÷ ({large[1]}) = {fmt(ratio)}")
print("Below 0.80: this flags a possible problem." if ratio < 0.8 else "0.80 or above: no flag.")
if not cfg["favorable"]:
    print("Remember: in COMPAS, being selected (predicted = 1) means being flagged as likely to be rearrested.")

In [ ]:
# @title Chart: rates by group, test rows
fig, ax = plt.subplots(figsize=(10.5, 4.5))
x = np.arange(len(RATES))
for i, (g, color) in enumerate(zip(GROUPS, [TEAL, ORANGE])):
    vals = bg.loc[g, RATES].to_numpy(dtype=float)
    bars = ax.bar(x + (i - 0.5) * 0.38, vals, 0.36, color=color)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{g}\n{fmt(v)}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x, RATES)
ax.set_ylim(0, 1.25)
ax.set_xlabel("rate (test rows)")
ax.set_ylabel("value (0 to 1)")
ax.set_title(f"{DATASET}: original model, by group")
ax.grid(axis="y", alpha=0.3)
plt.show()

**For your report:** Which fairness definition does the baseline model meet (gap ≤ 0.05): demographic parity, equal opportunity, equalized odds, or none?

## Part 4 · Reweighing (train rows)
Reweighing gives each train row a weight, so that rare combinations of outcome and group count more: weight = P(label) ÷ P(label | group). You compute every number yourself from the table below.

In [ ]:
# @title Train rows: group × label counts
group_line()
display(two_way(train))

Use the calculator for any division, for example (3/4) / (1/2). It prints the result to 4 decimals.

In [ ]:
# @title Calculator
EXPRESSION = "(3/4) / (1/2)"  # @param {type:"string"}
result = safe_calc(EXPRESSION)
print(result if isinstance(result, str) else f"{EXPRESSION} = {result:.4f}")

For each combination, type P(label | group), P(label) and the weight, each with 4 decimals. Then press ▶.

In [ ]:
# @title Your probabilities and weights
# @markdown **label = 1, Group A**
A1_P_label_given_group = 0.0  # @param {type:"number"}
A1_P_label = 0.0  # @param {type:"number"}
A1_weight = 0.0  # @param {type:"number"}
# @markdown **label = 0, Group A**
A0_P_label_given_group = 0.0  # @param {type:"number"}
A0_P_label = 0.0  # @param {type:"number"}
A0_weight = 0.0  # @param {type:"number"}
# @markdown **label = 1, Group B**
B1_P_label_given_group = 0.0  # @param {type:"number"}
B1_P_label = 0.0  # @param {type:"number"}
B1_weight = 0.0  # @param {type:"number"}
# @markdown **label = 0, Group B**
B0_P_label_given_group = 0.0  # @param {type:"number"}
B0_P_label = 0.0  # @param {type:"number"}
B0_weight = 0.0  # @param {type:"number"}

group_line()
for letter, g in zip("AB", GROUPS):
    for y in (1, 0):
        print(f"{letter}{y}_ fields: label = {y}, {g}")

Press ▶ to check every field. If all four weights are filled in, the cell also checks that each group's weighted rate of label = 1 equals P(label = 1).

In [ ]:
# @title Check your probabilities and weights
group_line()
exact = exact_weights()
p1 = train["label"].mean()
v = globals()
rows = []
for letter, g in zip("AB", GROUPS):
    for y in (1, 0):
        key = f"{letter}{y}"
        cond = train.loc[train[GROUP] == g, "label"].eq(y).mean()
        marg = p1 if y == 1 else 1 - p1
        rows.append({"combination": f"label = {y}, {g}",
                     f"{key}_P_label_given_group": number_check(v.get(f"{key}_P_label_given_group", 0.0), cond, 0.0001),
                     f"{key}_P_label": number_check(v.get(f"{key}_P_label", 0.0), marg, 0.0001),
                     f"{key}_weight": number_check(v.get(f"{key}_weight", 0.0), exact[(g, y)], 0.001)})
for r in rows:   # one small table per combination, so each field name is shown
    display(pd.DataFrame([r]).set_index("combination"))
w = your_audit_weights()
if any(x == 0 for x in w.values()):
    print("\nType your four weights first; then the weighted-rate check runs.")
else:
    print()
    print_check(w)

Press ▶ to retrain the model with your weights. The model never sees the group column.

In [ ]:
# @title Retrain with your weights, test rows
w = your_audit_weights()
pred_rw_yours = None
if any(x == 0 for x in w.values()):
    print("Type your four weights in Part 4 first.")
elif not all(ok for *_, ok in check_weights(w)):
    print("Your weights don't pass the weighted-rate check yet. Fix them, then press ▶ here again.")
else:
    pred_rw_yours = fit_reweighed(w)
    before = by_group(y_test, test["predicted"], test[GROUP])
    after = by_group(y_test, pred_rw_yours, test[GROUP])
    print("Before = original model (Part 3). After = your reweighed model. Test rows.")
    display(before_after(before, after))

**For your report:** Write the weight formula for one combination in words, then with your numbers. What does your weight do to that combination?

## Part 5 · Thresholds (test rows)
The model's scores stay the same; you choose one cutoff for each group. First choose which rate you will make equal in your scenario, then press ▶.

In [ ]:
# @title Choose the rate to equalize
RATE_TO_EQUALIZE = "TPR"  # @param ["selection rate", "TPR", "FPR"]

MEANING = {"selection rate": "how often each group gets predicted = 1",
           "TPR": "how often people who really have label = 1 get predicted = 1",
           "FPR": "how often people who really have label = 0 get predicted = 1"}
print(f"You will equalize the {RATE_TO_EQUALIZE}: {MEANING[RATE_TO_EQUALIZE]}.")

**For your report:** In plain words, what should be equal for the two groups in your scenario? Which rate measures it, and why is that the right rate here?

Move the sliders, then press ▶. Aim for a gap of 0.03 or less in the rate you chose.

In [ ]:
# @title Your cutoffs
cutoff_groupA = 0.50  # @param {type:"slider", min:0.01, max:0.99, step:0.01}
cutoff_groupB = 0.50  # @param {type:"slider", min:0.01, max:0.99, step:0.01}

group_line()
cuts = {GROUPS[0]: cutoff_groupA, GROUPS[1]: cutoff_groupB}
pred_game, bg_game, out = threshold_game(cuts)
display(out)

chosen = globals().get("RATE_TO_EQUALIZE", "TPR")
g = gap_size(bg_game, chosen)
print(f"Is the {chosen} gap 0.03 or less? {'yes' if g <= 0.03 else 'not yet'}, gap = {g:.3f}")

**For your report:** Record your two cutoffs. Which other gap changed, and in which direction?

## Part 6 · Comparison (test rows)
The original model, your reweighed model (Part 4) and your threshold model (Part 5), on the same test rows. Use this table and chart in the Results section of your report.

In [ ]:
# @title Compare your three models
MODELS, missing = your_models()
table = None
if missing:
    print("Finish these first, then press ▶ here again:", "; ".join(missing))
else:
    table, gaps, styled = compare(MODELS)
    display(styled)
    print(GAP_LEGEND)

In [ ]:
# @title Chart: the three gaps for the three models
if table is None:
    print("Run the comparison cell above first.")
else:
    fig = plot_gaps(gaps)
    plt.show()

In [ ]:
# @title Download the table and the chart
if table is None:
    print("Run the comparison cell above first.")
else:
    table.to_csv("comparison.csv", encoding="utf-8-sig")
    plot_gaps(gaps).savefig("comparison.png", dpi=200, bbox_inches="tight")
    plt.close("all")
    try:
        from google.colab import files
        files.download("comparison.csv")
        files.download("comparison.png")
        print("Downloading comparison.csv and comparison.png. Check your browser's downloads.")
    except ImportError:
        print("Not running in Colab. The files were saved here:")
        print(os.path.abspath("comparison.csv"))
        print(os.path.abspath("comparison.png"))

## Part 7 · Who bears the errors (test rows)
A false positive (FP) is a wrong predicted = 1 for someone with label = 0; a false negative (FN) is a wrong predicted = 0 for someone with label = 1. "Per 100" counts them out of every 100 people in the group who could get that error.

In [ ]:
# @title Who bears the errors
MODELS, missing = your_models()
if missing:
    print("Not yet included:", "; ".join(missing))
sc = SCENARIOS[DATASET]
rows, per100 = [], {}
for name, p in MODELS.items():
    per100[name] = {}
    for g in GROUPS:
        m = (test[GROUP] == g).to_numpy()
        c = counts(test[m], np.asarray(p)[m])
        fp100 = int(rnd(100 * c["FP"] / (c["FP"] + c["TN"]), 0))
        fn100 = int(rnd(100 * c["FN"] / (c["FN"] + c["TP"]), 0))
        per100[name][g] = fp100 if sc["harm"] == "FP" else fn100
        rows.append({"model": name, "group": g,
                     "false positives": c["FP"], "FP per 100 with label = 0": fp100,
                     "false negatives": c["FN"], "FN per 100 with label = 1": fn100})
display(pd.DataFrame(rows).set_index(["model", "group"]))

counted = ("false positives (people with label = 0 who got predicted = 1)" if sc["harm"] == "FP" else
           "false negatives (people with label = 1 who got predicted = 0)")
print(f"\nIn words, for the {counted}:")
A, B = GROUPS
LABELS = {"original": "Original model", "reweighed (yours)": "Your reweighed model",
          "threshold (yours)": "Your threshold model"}
for name in MODELS:
    print(f"{LABELS[name]}: {per100[name][A]} of every 100 {sc['who'][A]} and "
          f"{per100[name][B]} of every 100 {sc['who'][B]} {sc['harm_phrase']}.")

**For your report:** Which error is worse in your scenario, and which group bears more of it under each model?

## Before you submit
- [ ] Every **For your report** text cell has an answer.
- [ ] `comparison.png` is downloaded (Part 6) and in your report.
- [ ] The notebook is saved (File → Save) and shared: **File → Share → "Anyone with the link can view"**. Paste the link on the first page of your report.

*Datasets: Adult (Becker & Kohavi, 1996), German Credit (Hofmann, 1994) and Bank Marketing (Moro, Rita & Cortez, 2014), UCI Machine Learning Repository, CC BY 4.0; COMPAS data from ProPublica (Angwin et al., 2016). Library: Fairlearn (Weerts et al., 2023).*